# Caching & Cost Optimization: serve repeated work for free

Every query to an LLM app costs money and time — embed + retrieve + an LLM call ($/token + latency).
At scale a query stream is full of **repeats** and **paraphrases** ("how tall is the imager?" and
"imager height?" want the same answer), and a naive app re-pays the **full** cost for each. A
**semantic cache** serves the cached answer whenever a new query is close enough (by embedding cosine)
to one already answered — so repeated/similar work becomes nearly free. The saving is exactly
**hit_rate × per-query cost**.

This notebook builds a semantic cache from primitives on **CPU**, then measures it on a query stream —
importing the *same* canonical code the concept page and the `.py` use (`caching_cost.py`, which reuses
ch5's encoder and ch12's cost model). Every number here is the chapter's own, asserted before it's claimed.

**What is real vs illustrative** (stated up front, and again in the banner the first cell prints):
- **Real & measured:** the semantic cache (embed → nearest cached query by cosine → HIT/MISS at a
  threshold), the **hit rate** over a real query stream, and the **cost/latency** arithmetic (ch12's
  token cost model + a modelled per-call latency). Every score is computed and asserted.
- **Illustrative (labelled):** the answer *text* a MISS "computes" is a fixed template (no LLM here),
  and the exact **latency constants** are modelled. The cache mechanism and hit-rate/cost math are real.
- **Carried caveat:** the cache matches by cosine (topic, not exact intent), so a too-low threshold
  serves a **false hit** — the wrong cached answer. Shown explicitly below.


> **Step 1 — import the canonical code + print the reproducibility banner.** Everything comes from
> `caching_cost.py`; nothing is re-defined here. The banner prints `numpy`/`torch` versions and the
> accelerator so the run is reproducible (the encoder is CPU-pinned).

In [1]:
import numpy as np
import torch

from caching_cost import (
    BASE_ANSWERS,
    CACHE_THRESHOLD,
    QUERY_STREAM,
    STREAM_ANSWERS,
    DenseRetriever,
    SemanticCache,
    build_probes,
    expected_cost_per_query,
    false_hit_and_miss_rates,
    full_call_cost_usd,
    full_corpus,
    hit_cost_usd,
    run_stream,
    savings_fraction,
)

print("numpy:", np.__version__)
_acc = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("torch:", torch.__version__, "| accelerator available:", _acc, "| encoder runs on: cpu")

corpus = full_corpus()
dense = DenseRetriever(corpus)
print(f"corpus: {len(corpus)} passages | dense lens: {dense.backend} | cache threshold: {CACHE_THRESHOLD}")
print(f"cost model (ch12): full call ${full_call_cost_usd():.6f} | cache hit ${hit_cost_usd():.6f}")
print("NOTE: the semantic cache + hit rate + cost/latency math are REAL; only the MISS answer text and exact latency constants are illustrative stand-ins.")

numpy: 2.4.6
torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus: 11 passages | dense lens: sentence-transformers/all-MiniLM-L6-v2 | cache threshold: 0.8
cost model (ch12): full call $0.002280 | cache hit $0.000036
NOTE: the semantic cache + hit rate + cost/latency math are REAL; only the MISS answer text and exact latency constants are illustrative stand-ins.


## 1) A query stream through the semantic cache

The stream has 3 distinct questions plus repeats and paraphrases. The first time a distinct question
is seen it **misses** (cold cache) and is computed + stored; every repeat or near-paraphrase after
**hits** and is served from the cache.

> **Step 2 — run the stream and read each query's verdict.** A parallel cache reports the cosine
> for each lookup so you can see why each hit or missed.

In [2]:
result = run_stream(dense, QUERY_STREAM, STREAM_ANSWERS)
trace_cache = SemanticCache(dense)  # a parallel cache to report per-query cosines
for query, hit in zip(QUERY_STREAM, result.hits):
    lookup = trace_cache.lookup(query)
    if hit:
        verdict = f"HIT  (cos {lookup.best_cosine:.3f} >= {CACHE_THRESHOLD})"
    elif trace_cache.entries:
        verdict = f"MISS (cos {lookup.best_cosine:.3f} < {CACHE_THRESHOLD})"
    else:
        verdict = "MISS (cold cache)"
    print(f"{verdict:<32} {query}")
    if not hit:
        trace_cache.store(query, STREAM_ANSWERS[query])
print(f"\nhit rate: {sum(result.hits)}/{len(result.hits)} = {result.hit_rate:.3f}")

MISS (cold cache)                What is the ground resolution of the Helios-7 imager?
MISS (cos 0.577 < 0.8)           When was Helios-7 launched?
HIT  (cos 1.000 >= 0.8)          What is the ground resolution of the Helios-7 imager?
HIT  (cos 0.920 >= 0.8)          How fine is the Helios-7 imager's ground resolution?
MISS (cos 0.743 < 0.8)           Who is the Helios-7 project lead?
HIT  (cos 0.844 >= 0.8)          What date did Helios-7 lift off?
HIT  (cos 0.974 >= 0.8)          Who leads the Helios-7 project?
HIT  (cos 1.000 >= 0.8)          When was Helios-7 launched?

hit rate: 5/8 = 0.625


> **Step 3 — assert the hit pattern by value.** 3 distinct questions → 3 cold misses; the 5
> repeats/paraphrases hit. The exact pattern is pinned so a library drift can't silently change it.

In [3]:
assert sum(result.hits) == 5, "5 of the 8 stream queries are repeats/paraphrases -> hits"
assert result.hit_rate == 5 / 8, "hit rate is 5/8 on this stream"
assert result.hits == (False, False, True, True, False, True, True, True), "the exact hit pattern (cold misses first)"
print("asserted: 3 distinct questions miss once (cold); every repeat/paraphrase after is a HIT (5/8 = 0.625).")

asserted: 3 distinct questions miss once (cold); every repeat/paraphrase after is a HIT (5/8 = 0.625).


![The semantic-cache mechanism: query → embed → nearest cached query (cosine ≥ τ ? HIT : MISS →
compute + store). Matches the page's Mermaid diagram. Generated by `make_figures_16.py`.](../../images/rag16_cache_flow.png)

![Animated — queries streaming in: hits served instantly (green), misses computing + filling the cache
(red), the hit rate climbing to 5/8. Generated by `make_animation_16.py`.](../../images/rag16_cache_stream.gif)

## 2) The payoff — cost and latency saved

The saving is **hit_rate × per-query cost**: a hit pays only the tiny lookup-embedding cost, a miss
pays the full call. The expected per-query cost is `(1−h)·full + h·hit`.

> **Step 4 — measure cost and latency saved on the stream, and check the expected-cost identity.**
> The measured cached cost must equal `n × expected_cost_per_query(hit_rate)`.

In [4]:
print(f"cost   : no-cache ${result.cost_no_cache_usd:.6f}  ->  cached ${result.cost_cached_usd:.6f}  "
      f"(saved {result.cost_saved_usd / result.cost_no_cache_usd:.1%})")
print(f"latency: no-cache {result.latency_no_cache_ms:.0f}ms  ->  cached {result.latency_cached_ms:.0f}ms  "
      f"(saved {result.latency_saved_ms / result.latency_no_cache_ms:.1%})")
predicted = len(QUERY_STREAM) * expected_cost_per_query(result.hit_rate)
print(f"identity: {len(QUERY_STREAM)} x expected_cost_per_query({result.hit_rate:.3f}) = ${predicted:.6f} == measured ${result.cost_cached_usd:.6f}")
assert abs(predicted - result.cost_cached_usd) < 1e-9, "the expected-cost identity must match the measured cost"
assert abs(savings_fraction(result.hit_rate) - result.hit_rate) < 0.02, "savings fraction ~ hit rate (a hit is nearly free)"
print(f"\nasserted: savings ~ hit rate — at {result.hit_rate:.1%} hits you cut ~{savings_fraction(result.hit_rate):.1%} of the bill.")

cost   : no-cache $0.018240  ->  cached $0.007020  (saved 61.5%)
latency: no-cache 6400ms  ->  cached 2425ms  (saved 62.1%)
identity: 8 x expected_cost_per_query(0.625) = $0.007020 == measured $0.007020

asserted: savings ~ hit rate — at 62.5% hits you cut ~61.5% of the bill.


![Expected per-query cost falls linearly with the hit rate; the stream's operating point (hit rate
0.625 → 61.5% saved) is marked. Generated by `make_figures_16.py`.](../../images/rag16_cost_vs_hitrate.png)

![Latency: a cache hit (5 ms) vs a full call (800 ms), and the stream's total (6400 ms → 2425 ms, 62%
saved). Generated by `make_figures_16.py`.](../../images/rag16_latency.png)

## 3) The false-hit / miss tradeoff

The threshold τ is a dial. **Lower** τ catches more paraphrases (fewer misses) but risks a **false
hit** — serving a genuinely *different* query the wrong cached answer. **Higher** τ is safe but misses
real paraphrases. A false hit is the dangerous error: it serves a confidently wrong answer.

> **Step 5 — sweep τ over labelled probes and watch the two error rates trade.** false-hit falls
> as τ rises; miss (of should-hit paraphrases) rises.

In [5]:
probes = build_probes()
print(f"{'threshold':>9} | {'false-hit':>9} | {'miss (of should-hit)':>21}")
print("-" * 46)
rows = []
for tau in (0.50, 0.60, 0.70, 0.80, 0.90):
    fh, miss = false_hit_and_miss_rates(dense, probes, tau)
    rows.append((tau, fh, miss))
    print(f"{tau:>9.2f} | {fh:>9.3f} | {miss:>21.3f}")
fhs = [fh for _, fh, _ in rows]
misses = [m for _, _, m in rows]
assert fhs == sorted(fhs, reverse=True), "false-hit must be non-increasing as tau rises"
assert misses == sorted(misses), "miss must be non-decreasing as tau rises"
assert fhs[0] > fhs[-1], "a low threshold produces MORE false hits than a high one (the tradeoff is real)"
print("\nasserted: raising tau cuts false hits but misses more paraphrases — the semantic-cache tradeoff.")

threshold | false-hit |  miss (of should-hit)
----------------------------------------------


     0.50 |     1.000 |                 0.000


     0.60 |     0.667 |                 0.000


     0.70 |     0.667 |                 0.000


     0.80 |     0.667 |                 0.000


     0.90 |     0.333 |                 0.333

asserted: raising tau cuts false hits but misses more paraphrases — the semantic-cache tradeoff.


![The false-hit / miss tradeoff as τ sweeps: false-hit (wrong answer served) falls, miss (real
paraphrase not cached) rises. The default τ=0.8 is marked. Generated by `make_figures_16.py`.](../../images/rag16_tradeoff.png)

![The cache-layers stack: semantic / prompt (prefix) / embedding / retrieval caching, each cutting a
different cost. Generated by `make_figures_16.py`.](../../images/rag16_cache_layers.png)

## 4) A false hit, concretely

At a **low** threshold, a query about a *different* imager attribute (wavelength, not resolution) is
close enough to a cached query to be wrongly served its answer. At the **default** threshold it
correctly misses.

> **Step 6 — a wavelength query wrongly hits the resolution answer at τ=0.5, but correctly misses
> at τ=0.8.** cosine matches topic, not exact intent.

In [6]:
cached_q = "What is the ground resolution of the Helios-7 imager?"
danger = "What is the wavelength range of the Helios-7 imager?"  # DIFFERENT imager attribute

low = SemanticCache(dense, threshold=0.50)
low.store(cached_q, BASE_ANSWERS[cached_q])
low_lookup = low.lookup(danger)
print(f"at tau=0.50: cosine {low_lookup.best_cosine:.3f} -> {'FALSE HIT' if low_lookup.hit else 'correct miss'}")
if low_lookup.hit:
    print(f"  served (WRONG): '{low_lookup.answer}'  <- answers RESOLUTION, not WAVELENGTH")

strict = SemanticCache(dense, threshold=CACHE_THRESHOLD)
strict.store(cached_q, BASE_ANSWERS[cached_q])
strict_lookup = strict.lookup(danger)
print(f"at tau={CACHE_THRESHOLD}: cosine {strict_lookup.best_cosine:.3f} -> {'FALSE HIT' if strict_lookup.hit else 'correct MISS'}")
assert low_lookup.hit, "at tau=0.50 the wavelength query wrongly hits the resolution answer"
assert not strict_lookup.hit, f"at tau={CACHE_THRESHOLD} the different-intent query correctly misses"
print("\nasserted: a low tau serves a WRONG cached answer; a high tau (near-duplicates only) fixes it.")

at tau=0.50: cosine 0.757 -> FALSE HIT
  served (WRONG): 'The Helios-7 imager has a ground resolution of 4 meters.'  <- answers RESOLUTION, not WAVELENGTH
at tau=0.8: cosine 0.757 -> correct MISS

asserted: a low tau serves a WRONG cached answer; a high tau (near-duplicates only) fixes it.


## Try it yourself

Before you run the next cell, **predict**. So far the stream hit rate was **5/8 = 0.625**. Now imagine
a **heavier-repeat** traffic pattern — the *same* stream but with each of the 3 distinct questions
asked one extra time at the end (3 more queries, all repeats → all hits).

**Question:** what will the new hit rate be — higher or lower than 0.625? And roughly what fraction of
the bill will the cache now save?

Think, then run.

In [7]:
# the same stream + one extra repeat of each of the 3 distinct base questions (all HITS)
heavier_stream = QUERY_STREAM + tuple(BASE_ANSWERS.keys())
heavier = run_stream(dense, heavier_stream, STREAM_ANSWERS)
print(f"stream length: {len(QUERY_STREAM)} -> {len(heavier_stream)}")
print(f"hit rate: {sum(heavier.hits)}/{len(heavier.hits)} = {heavier.hit_rate:.3f}  (was {result.hit_rate:.3f})")
print(f"cost saved: {heavier.cost_saved_usd / heavier.cost_no_cache_usd:.1%}  (was {result.cost_saved_usd / result.cost_no_cache_usd:.1%})")

stream length: 8 -> 11
hit rate: 8/11 = 0.727  (was 0.625)
cost saved: 71.6%  (was 61.5%)


> **Was your prediction right?** More repeats → a **higher** hit rate → a bigger saving. The 3 extra
> queries are all repeats of already-cached questions, so they all hit: the hit count rises from 5 to 8
> out of 11. The lesson is the whole economics of caching: **the payoff scales with how repetitive your
> traffic is.** High-repeat traffic (FAQs, popular queries, agent loops re-asking the same thing) is
> where a cache pays off hugely; a stream of all-unique queries saves nothing. The next cell asserts the
> higher hit rate so the notebook stays honest about it.

In [8]:
assert heavier.hit_rate > result.hit_rate, "more repeats -> a higher hit rate"
assert sum(heavier.hits) == 8, "the 3 extra queries are all repeats -> 8 hits out of 11"
assert heavier.hit_rate == 8 / 11, "hit rate is 8/11 on the heavier-repeat stream"
print(f"asserted: hit rate {result.hit_rate:.3f} -> {heavier.hit_rate:.3f}; caching pays off with repetitive traffic.")

asserted: hit rate 0.625 -> 0.727; caching pays off with repetitive traffic.


## What we saw — and the RAG domain, tied together

- **Repeats and paraphrases are free.** The semantic cache turned 5 of 8 stream queries into hits
  (**0.625** hit rate), cutting **61.5%** of cost and **62%** of latency — the saving is `hit_rate ×
  per-query cost`.
- **The threshold is a false-hit / miss dial.** A low τ serves a confidently **wrong** cached answer
  (cosine ≈ topic, not intent); a high τ is safe but misses paraphrases. τ=0.8 caught the paraphrases
  while correctly missing the wavelength-vs-resolution near-duplicate.
- **Caching pays off with repetitive traffic.** The heavier-repeat stream lifted the hit rate to
  **8/11** — the more your traffic repeats, the more a cache saves; all-unique traffic saves nothing.
- **Cache at every layer:** semantic (whole-answer), prompt/prefix (provider KV reuse), embedding
  (don't re-embed), retrieval (reuse retrieved chunks) — they compose.

**The thread through all 16 RAG chapters** is **cost, latency, and quality**. Chunking sized the
retrievable unit; embeddings and retrieval found the right context cheaply; re-ranking spent compute
only on the top-k; long-context-vs-RAG weighed stuffing's cost against retrieval's; orchestration wired
the steps; guardrails protected quality; and caching — this chapter — makes the whole thing affordable
at scale. Every chapter was, in the end, about getting the *right context* to the model for the *least
cost* without sacrificing *quality*. That is RAG.

**Provenance:** semantic caching — **GPTCache** (Bang 2023, NLP-OSS); prompt/prefix caching — Anthropic
`cache_control` + OpenAI automatic prompt caching + **Prompt Cache** (Gim et al. 2023, arXiv:2311.04934);
the cost model reuses ch12. All in the references file.
